
# 練習問題: モンテカルロ法で π を求める (reduction)

## 目標

`reduction(+:count)` 句を1つ追加するだけで, 複数のスレッドが同じカウンタ `count` を同時に増やそうとして起こる競合 (data race) を解消し,
モンテカルロ法による円周率 π の推定を正しく並列化できることを体験する.

## 課題

`montecarlo_pi.cpp` (または `montecarlo_pi.f90`) は, 単位正方形 [0,1)×[0,1) に n 個の点をランダムに投げ,
原点からの距離が 1 未満 (= 半径 1 の円の 1/4 の内側) に入った点の数 `count` を数えて, π ≈ 4 * count / n を求めるコードである.
乱数は反復番号から決まる線形合同法 (LCG) で生成しているので, スレッド数によらず同じ点列になる.

このループを単純に並列化すると, 全スレッドが共有変数 `count` に同時に `count++` を行うため競合 (data race) が起き,
スレッド数が2以上だと**数え落とし**が発生して π が小さめに出たり, 実行ごとに値が変わったりする.

コメント `TODO` の箇所で **`reduction` を用いて並列化せよ**. 各スレッドが部分カウントを持ち寄って正しい合計を得るようになる.

- C++: `#pragma omp parallel for reduction(+:count)`
- Fortran: `!$omp parallel do private(x, y) reduction(+:count)` … `!$omp end parallel do`

それ以外のコードを変更する必要はない.

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore montecarlo_pi.cpp -o montecarlo_pi.exe

# Fortran
nvfortran -fast -mp=multicore montecarlo_pi.f90 -o montecarlo_pi.exe
```

```
OMP_NUM_THREADS=4 ./montecarlo_pi.exe
```

第1引数で点数 `n` を変えられる (例: `./montecarlo_pi.exe 1000000000`).

## 期待される結果

点数 n を大きくするほど, π の推定値は真の値 3.14159... に近づく.

```
pi ~= 3.141...
```

- `reduction(+:count)` を**追加すると**, スレッド数によらず常に同じ正しい値になる.
- 参考: もし `reduction` を付けずに共有変数 `count` をそのまま `count++` で並列に更新すると (例: `OMP_NUM_THREADS=4`),
  複数スレッドの加算がぶつかって数え落としが起き, `count` が本来より小さくなって π が 3.14 より小さく出たり, 実行ごとに値が変わったりする.
  これが reduction が必要な理由である.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ montecarlo_pi.cpp
#include <cstdio>
#include <cstdlib>

/* 状態を持たない (カウンタベースの) 乱数: (seed,k) から [0,1) の値を決める純粋関数。
   - seed は「どの乱数列(ストリーム)か」, k は「その何番目か」。同じ (seed,k) なら必ず同じ値。
   - 毎回 (seed,k) から計算し共有状態を持たないので, 並列化しても引かれる乱数列は
     スレッド数によらず同一になる (競合も起きない)。
   (教育用の簡単なハッシュ。M=2^31-1 未満で計算し, 途中の積も 64bit に収まる。)        */
static inline double draw_rand01(long long seed, long long k) {
  const long long M = 2147483647LL;   /* 2^31 - 1 */
  long long x = ((seed % M) * 2654435761LL + (k % M) + 1) % M;
  x = ((x ^ (x >> 16)) * 1812433253LL) % M;
  x = ((x ^ (x >> 13)) * 1664525LL)    % M;
  x =  (x ^ (x >> 16)) % M;
  return (double)x / (double)M;        /* [0,1) */
}

int main(int argc, char ** argv) {
  long n = (1 < argc ? atol(argv[1]) : 100L * 1000L * 1000L);
  long count = 0;               /* 単位円の 1/4 の内側に入った点数 */
  printf("n = %ld\n", n);
  /* 単位正方形 [0,1)x[0,1) に n 点を投げ, 半径 1 の円の内側に入った点を数える。
     点 i は乱数列 i の 0,1 番目を x,y 座標に使う。 */
  // TODO: 円内に入った点数を reduction(+:count) で集計して π を求めよ.
  for (long i = 0; i < n; i++) {
    double x = draw_rand01(i, 0);
    double y = draw_rand01(i, 1);
    if (x * x + y * y < 1.0) count++;
  }
  double pi = 4.0 * (double)count / (double)n;
  printf("count = %ld / %ld\n", count, n);
  printf("pi ~= %.6f\n", pi);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore montecarlo_pi.cpp -o montecarlo_pi_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./montecarlo_pi_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ montecarlo_pi.f90
module rng_mod
contains
  ! 状態を持たない (カウンタベースの) 乱数: (seed,k) から [0,1) の値を決める純粋関数。
  ! 同じ (seed,k) なら必ず同じ値 → 並列化しても引かれる乱数列はスレッド数によらず同一。
  ! (教育用の簡単なハッシュ。M=2^31-1 未満で計算し, 途中の積も 64bit に収まる。)
  function draw_rand01(seed, k) result(u)
    integer(8), intent(in) :: seed, k
    real(8) :: u
    integer(8), parameter :: M = 2147483647_8   ! 2^31 - 1
    integer(8) :: x
    x = modulo(modulo(seed, M) * 2654435761_8 + modulo(k, M) + 1_8, M)
    x = modulo(ieor(x, ishft(x, -16)) * 1812433253_8, M)
    x = modulo(ieor(x, ishft(x, -13)) * 1664525_8,    M)
    x = modulo(ieor(x, ishft(x, -16)), M)
    u = real(x, 8) / real(M, 8)        ! [0,1)
  end function draw_rand01
end module rng_mod

program montecarlo_pi
  use rng_mod
  character(len=64) :: arg
  integer(8) :: n, i, count
  real(8) :: x, y, pi
  n = 100_8 * 1000_8 * 1000_8
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) n
  end if
  count = 0                      ! 単位円の 1/4 の内側に入った点数
  print "(a,i0)", "n = ", n
  ! 単位正方形 [0,1)x[0,1) に n 点を投げ, 半径 1 の円の内側に入った点を数える。
  ! 点 i は乱数列 i の 0,1 番目を x,y 座標に使う。
  ! TODO: 円内に入った点数を reduction(+:count) で集計して π を求めよ.
  do i = 0, n - 1
     x = draw_rand01(i, 0_8)
     y = draw_rand01(i, 1_8)
     if (x * x + y * y < 1.0d0) count = count + 1
  end do
  ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do).
  pi = 4.0d0 * real(count, 8) / real(n, 8)
  print "(a,i0,a,i0)", "count = ", count, " / ", n
  print "(a,f0.6)", "pi ~= ", pi
end program montecarlo_pi

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore montecarlo_pi.f90 -o montecarlo_pi_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./montecarlo_pi_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:montecarlo_pi.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:montecarlo_pi.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:montecarlo_pi.cpp}